# Stress Prediction v5 — Native CatBoost Ensemble

이번 버전의 핵심:

- 기존 성능이 좋았던 ExtraTrees 구조는 그대로 유지
- CatBoost는 One-Hot Encoding 없이 범주형 변수를 문자열 그대로 사용
- CatBoost의 `RMSE`, `MAE`, `Quantile` 목적함수를 각각 비교
- 모든 모델의 OOF MAE와 상관계수 확인
- CV 최적 앙상블과 CatBoost를 강제로 포함한 후보 제출 파일 생성
- 반올림과 최근접 이웃 override를 함께 검증

> 먼저 아래 경로 설정만 사용자 환경에 맞게 확인한 뒤 위에서부터 실행하세요.


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_SPLITS = 5
N_ESTIMATORS = 1200

TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.48

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

# 사용자 환경에 맞게 확인
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CatBoost 사용 가능:", CATBOOST_AVAILABLE)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

if not CATBOOST_AVAILABLE:
    print("\nCatBoost가 없다면 아래 명령을 별도 셀에서 실행한 뒤 커널을 재시작하세요.")
    print("!pip install catboost")


CatBoost 사용 가능: True
DATA_DIR: /Users/bitsaem/Desktop/stress_project/open
OUTPUT_DIR: /Users/bitsaem/Desktop/stress_project/8_6


## 1. 공통 전처리와 기존 ExtraTrees/Pair 함수


In [2]:
def compute_iqr_bounds(train_frame: pd.DataFrame, columns: list[str]) -> dict:
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = result["weight"] / (height_m.pow(2) + 1e-6)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = frame.isna().sum(axis=1)
    return result


def add_catboost_features(frame: pd.DataFrame) -> pd.DataFrame:
    # CatBoost에는 과도한 파생변수 대신 안정적인 변수만 추가
    result = add_base_features(frame)

    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bp_sum"] = sbp + dbp
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_working"] = result["age"] * result["mean_working"]
    result["working_bmi"] = result["mean_working"] * result["bmi"]
    result["cholesterol_glucose_product"] = (
        result["cholesterol"] * result["glucose"]
    )

    # 연속형 구간 범주: CatBoost가 상호작용을 찾는 데 도움을 줄 수 있음
    result["age_band"] = pd.cut(
        result["age"],
        bins=[0, 35, 50, 65, 80, np.inf],
        labels=["age_1", "age_2", "age_3", "age_4", "age_5"],
    ).astype("object")

    result["bmi_band"] = pd.cut(
        result["bmi"],
        bins=[-np.inf, 18.5, 23, 25, 30, np.inf],
        labels=["bmi_1", "bmi_2", "bmi_3", "bmi_4", "bmi_5"],
    ).astype("object")

    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_dense_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="constant", fill_value="missing"),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(strategy="median", add_indicator=True),
            numerical,
        ),
    ])


def extra_trees_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    # 원본 모델과 동일하게 gender 제거 및 feature copy 유지
    train_no_gender = train_features.drop(columns=["gender"], errors="ignore")
    val_no_gender = val_features.drop(columns=["gender"], errors="ignore")

    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)

    preprocessor = make_dense_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    return np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)

    sorted_train = np.sort(train_values)
    train_rank = (
        np.searchsorted(sorted_train, train_values, side="right")
        / len(train_values)
    )
    test_rank = (
        np.searchsorted(sorted_train, test_values, side="right")
        / len(train_values)
    )
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    train_rank, val_rank = {}, {}

    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    pair_targets = []

    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argmin(distance, axis=1)
        pair_targets.append(target_values[nearest_idx])

    return np.column_stack(pair_targets)


def nearest_neighbor_lookup(
    train_raw: pd.DataFrame,
    val_raw: pd.DataFrame,
    target: pd.Series,
    numeric_cols: list[str],
):
    train_numeric = train_raw[numeric_cols].copy()
    val_numeric = val_raw[numeric_cols].copy()

    medians = train_numeric.median()
    train_numeric = train_numeric.fillna(medians)
    val_numeric = val_numeric.fillna(medians)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)

    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


## 2. CatBoost 전용 데이터 처리와 모델


In [3]:
def prepare_native_catboost_frames(
    train_frame: pd.DataFrame,
    val_frame: pd.DataFrame,
):
    train_cb = train_frame.copy()
    val_cb = val_frame.copy()

    categorical_cols = train_cb.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    # CatBoost의 범주형 변수는 문자열이어야 하므로 결측값도 문자열로 변환
    for col in categorical_cols:
        train_cb[col] = train_cb[col].fillna("__MISSING__").astype(str)
        val_cb[col] = val_cb[col].fillna("__MISSING__").astype(str)

    # 수치 결측값은 fold별 train 중앙값 사용
    numeric_cols = [
        col for col in train_cb.columns
        if col not in categorical_cols
    ]
    medians = train_cb[numeric_cols].median()

    train_cb[numeric_cols] = train_cb[numeric_cols].fillna(medians)
    val_cb[numeric_cols] = val_cb[numeric_cols].fillna(medians)

    return train_cb, val_cb, categorical_cols


CATBOOST_CONFIGS = {
    # MAE 평가에서는 RMSE 목적함수가 더 안정적으로 학습되는 경우가 많음
    "cat_rmse": {
        "loss_function": "RMSE",
        "eval_metric": "MAE",
        "iterations": 2500,
        "learning_rate": 0.025,
        "depth": 6,
        "l2_leaf_reg": 8.0,
        "random_strength": 0.8,
        "bagging_temperature": 0.5,
    },
    "cat_mae": {
        "loss_function": "MAE",
        "eval_metric": "MAE",
        "iterations": 2000,
        "learning_rate": 0.025,
        "depth": 6,
        "l2_leaf_reg": 8.0,
        "random_strength": 0.5,
        "bagging_temperature": 0.5,
    },
    # 기존 최적 tree quantile 근처의 조건부 분위수 예측
    "cat_quantile": {
        "loss_function": "Quantile:alpha=0.52",
        "eval_metric": "MAE",
        "iterations": 2200,
        "learning_rate": 0.025,
        "depth": 6,
        "l2_leaf_reg": 8.0,
        "random_strength": 0.5,
        "bagging_temperature": 0.5,
    },
}


def catboost_prediction(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    config_name: str,
    eval_target: pd.Series | None = None,
):
    if not CATBOOST_AVAILABLE:
        raise ImportError(
            "CatBoost가 설치되어 있지 않습니다. !pip install catboost 실행 후 재시작하세요."
        )

    train_cb, val_cb, categorical_cols = prepare_native_catboost_frames(
        train_features,
        val_features,
    )

    params = CATBOOST_CONFIGS[config_name].copy()

    model = CatBoostRegressor(
        **params,
        random_seed=RANDOM_SEED,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
        od_type="Iter",
        od_wait=150,
    )

    fit_kwargs = {
        "X": train_cb,
        "y": target,
        "cat_features": categorical_cols,
        "verbose": False,
    }

    # 검증 데이터를 전달하여 early stopping 사용
    if eval_target is not None:
        fit_kwargs["eval_set"] = (val_cb, eval_target)
        fit_kwargs["use_best_model"] = True

    model.fit(**fit_kwargs)
    return model.predict(val_cb), model.get_best_iteration()


## 3. 데이터 로드 및 OOF 생성


In [4]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
submission_path = DATA_DIR / "sample_submission.csv"

# 현재 첨부 파일로 실행할 때의 fallback
if not train_path.exists():
    fallback = Path("/mnt/data/train(2).csv")
    if fallback.exists():
        train_path = fallback

train_raw = pd.read_csv(train_path)

required_columns = {
    TARGET,
    ID_COLUMN,
    "gender",
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
    "activity",
    "smoke_status",
    "medical_history",
    "family_medical_history",
    "sleep_pattern",
    "edu_level",
    "mean_working",
}

missing_columns = required_columns - set(train_raw.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")

print("Train shape:", train_raw.shape)
display(train_raw.head())
display(train_raw.dtypes.to_frame("dtype"))

y = train_raw[TARGET].reset_index(drop=True)
raw_input = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()


Train shape: (3000, 18)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


,dtype
ID,object
gender,object
age,int64
height,float64
weight,float64
cholesterol,float64
systolic_blood_pressure,int64
diastolic_blood_pressure,int64
glucose,float64
bone_density,float64


In [5]:
if not CATBOOST_AVAILABLE:
    raise ImportError(
        "CatBoost 설치가 필요합니다. 새 셀에서 !pip install catboost 실행 후 커널을 재시작하세요."
    )

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

model_names = [
    "extra_trees",
    "pair_k1",
    "cat_rmse",
    "cat_mae",
    "cat_quantile",
]

oof = {
    name: np.zeros(len(y), dtype=float)
    for name in model_names
}

nearest_dist_oof = np.zeros(len(y), dtype=float)
nearest_target_oof = np.zeros(len(y), dtype=float)

best_iterations = {
    name: []
    for name in ["cat_rmse", "cat_mae", "cat_quantile"]
}

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_fold_raw = raw_input.iloc[train_idx].reset_index(drop=True)
    val_fold_raw = raw_input.iloc[val_idx].reset_index(drop=True)

    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    # ET와 Pair용 winsorize
    bounds = compute_iqr_bounds(train_fold_raw, WINSORIZE_COLUMNS)
    train_w = apply_winsorize(train_fold_raw, bounds)
    val_w = apply_winsorize(val_fold_raw, bounds)

    train_base = add_base_features(train_w)
    val_base = add_base_features(val_w)

    et_matrix = extra_trees_prediction_matrix(
        train_base,
        val_base,
        y_train,
    )
    pair_matrix = pair_prediction_matrix(
        train_base,
        val_base,
        y_train,
    )

    oof["extra_trees"][val_idx] = np.quantile(
        et_matrix,
        TREE_QUANTILE,
        axis=1,
    )
    oof["pair_k1"][val_idx] = np.quantile(
        pair_matrix,
        PAIR_QUANTILE,
        axis=1,
    )

    # CatBoost에는 winsorize하지 않은 원본 기반 파생변수 사용
    train_cat = add_catboost_features(train_fold_raw)
    val_cat = add_catboost_features(val_fold_raw)

    for config_name in ["cat_rmse", "cat_mae", "cat_quantile"]:
        pred, best_iter = catboost_prediction(
            train_cat,
            val_cat,
            y_train,
            config_name=config_name,
            eval_target=y_val,
        )
        oof[config_name][val_idx] = pred
        best_iterations[config_name].append(best_iter)

    nearest_dist, nearest_target = nearest_neighbor_lookup(
        train_fold_raw,
        val_fold_raw,
        y_train,
        numeric_cols,
    )
    nearest_dist_oof[val_idx] = nearest_dist
    nearest_target_oof[val_idx] = nearest_target

    print(f"Fold {fold}/{N_SPLITS} 완료")


Fold 1/5 완료
Fold 2/5 완료
Fold 3/5 완료
Fold 4/5 완료
Fold 5/5 완료


## 4. 단일 모델 성능과 상관관계


In [6]:
score_rows = []

for name, pred in oof.items():
    clipped = np.clip(pred, 0, 1)
    score_rows.append({
        "model": name,
        "raw_mae": mean_absolute_error(y, clipped),
        "rounded_2_mae": mean_absolute_error(
            y,
            np.round(clipped, 2),
        ),
        "pred_mean": clipped.mean(),
        "pred_std": clipped.std(),
    })

score_df = (
    pd.DataFrame(score_rows)
    .sort_values("raw_mae")
    .reset_index(drop=True)
)

print("모델별 OOF MAE")
display(score_df)

print("CatBoost fold별 best iteration")
for name, values in best_iterations.items():
    print(
        name,
        values,
        "평균:",
        round(float(np.mean(values)), 1),
    )

print("\nOOF 예측 상관계수")
display(pd.DataFrame(oof).corr().round(4))


모델별 OOF MAE


,model,raw_mae,rounded_2_mae,pred_mean,pred_std
0,extra_trees,0.147479,0.147493,0.487475,0.190507
1,pair_k1,0.154009,0.153980,0.464737,0.202879
2,cat_rmse,0.214863,0.214870,0.478509,0.115083
3,cat_quantile,0.217061,0.217107,0.493211,0.109774
4,cat_mae,0.218730,0.218743,0.474396,0.107370


CatBoost fold별 best iteration
cat_rmse [2499, 2497, 2496, 2499, 2498] 평균: 2497.8
cat_mae [1990, 1999, 1999, 1982, 1995] 평균: 1993.0
cat_quantile [2193, 2177, 2199, 2198, 2189] 평균: 2191.2

OOF 예측 상관계수


,extra_trees,pair_k1,cat_rmse,cat_mae,cat_quantile
extra_trees,1.0000,0.8915,0.6618,0.5954,0.5997
pair_k1,0.8915,1.0000,0.6068,0.5476,0.5495
cat_rmse,0.6618,0.6068,1.0000,0.8921,0.8985
cat_mae,0.5954,0.5476,0.8921,1.0000,0.9401
cat_quantile,0.5997,0.5495,0.8985,0.9401,1.0000


## 5. 앙상블 탐색

두 종류의 결과를 만듭니다.

1. **CV 최적 조합**: OOF MAE가 가장 낮은 조합
2. **강제 다양성 조합**: CatBoost가 CV에서 불리해도 10~30% 포함한 조합

퍼블릭 점수와 로컬 CV의 상관이 약할 수 있으므로 두 종류 모두 저장합니다.


In [7]:
def apply_postprocess(
    prediction: np.ndarray,
    round_decimals: int | None,
    override_threshold: float | None,
    nearest_dist: np.ndarray,
    nearest_target: np.ndarray,
):
    result = np.clip(prediction.copy(), 0, 1)

    if round_decimals is not None:
        result = np.round(result, round_decimals)

    if override_threshold is not None:
        mask = nearest_dist < override_threshold
        replacement = nearest_target[mask].copy()

        if round_decimals is not None:
            replacement = np.round(replacement, round_decimals)

        result[mask] = np.clip(replacement, 0, 1)

    return result


# 성능이 가장 좋은 CatBoost 한 개를 자동 선택
cat_candidates = ["cat_rmse", "cat_mae", "cat_quantile"]
best_cat_name = min(
    cat_candidates,
    key=lambda name: mean_absolute_error(
        y,
        np.clip(oof[name], 0, 1),
    ),
)

print("선택된 CatBoost:", best_cat_name)

search_model_names = [
    "extra_trees",
    best_cat_name,
    "pair_k1",
]

pred_matrix = np.column_stack([
    oof[name]
    for name in search_model_names
])

search_rows = []

# ET, CatBoost, Pair 가중치를 0.05 간격으로 탐색
grid = np.arange(0.0, 1.0001, 0.05)

for et_w in grid:
    for cat_w in grid:
        pair_w = 1.0 - et_w - cat_w

        if pair_w < -1e-9:
            continue

        pair_w = max(pair_w, 0.0)
        weights = np.array([et_w, cat_w, pair_w])
        blended = pred_matrix @ weights

        for decimals in [2, None]:
            for threshold in [0.12, 0.15, 0.18, 0.20, 0.22, None]:
                final_pred = apply_postprocess(
                    blended,
                    decimals,
                    threshold,
                    nearest_dist_oof,
                    nearest_target_oof,
                )
                mae = mean_absolute_error(y, final_pred)

                search_rows.append({
                    "mae": mae,
                    "round_decimals": decimals,
                    "override_threshold": threshold,
                    "extra_trees": et_w,
                    best_cat_name: cat_w,
                    "pair_k1": pair_w,
                })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("mae")
    .reset_index(drop=True)
)

display(search_df.head(30))
print("최적 OOF MAE:", search_df.iloc[0]["mae"])


선택된 CatBoost: cat_rmse


,mae,round_decimals,override_threshold,extra_trees,cat_rmse,pair_k1
0,0.147087,2.0,0.20,0.85,0.0,0.15
1,0.147087,2.0,0.22,0.85,0.0,0.15
2,0.147087,2.0,0.12,0.85,0.0,0.15
3,0.147087,2.0,0.15,0.85,0.0,0.15
4,0.147087,2.0,0.18,0.85,0.0,0.15
5,0.147093,2.0,0.18,0.80,0.0,0.20
6,0.147093,2.0,0.22,0.80,0.0,0.20
7,0.147093,2.0,0.12,0.80,0.0,0.20
8,0.147093,2.0,0.20,0.80,0.0,0.20
9,0.147093,2.0,0.15,0.80,0.0,0.20


최적 OOF MAE: 0.14708666666666667


## 6. 제출 후보 구성


In [8]:
def row_to_config(row) -> dict:
    return {
        "weights": {
            "extra_trees": float(row["extra_trees"]),
            best_cat_name: float(row[best_cat_name]),
            "pair_k1": float(row["pair_k1"]),
        },
        "round_decimals": (
            None
            if pd.isna(row["round_decimals"])
            else int(row["round_decimals"])
        ),
        "override_threshold": (
            None
            if pd.isna(row["override_threshold"])
            else float(row["override_threshold"])
        ),
        "oof_mae": float(row["mae"]),
    }


configs = []

# 1. CV 최적
configs.append(("cv_best", row_to_config(search_df.iloc[0])))

# 2. CatBoost 10~15% 포함 최상위
candidate = search_df[
    (search_df[best_cat_name] >= 0.10)
    & (search_df[best_cat_name] <= 0.15)
]
if len(candidate):
    configs.append(("cat_10_15", row_to_config(candidate.iloc[0])))

# 3. CatBoost 20~25% 포함 최상위
candidate = search_df[
    (search_df[best_cat_name] >= 0.20)
    & (search_df[best_cat_name] <= 0.25)
]
if len(candidate):
    configs.append(("cat_20_25", row_to_config(candidate.iloc[0])))

# 4. 강제 균형 후보
forced_configs = [
    (
        "forced_cat15",
        {
            "weights": {
                "extra_trees": 0.70,
                best_cat_name: 0.15,
                "pair_k1": 0.15,
            },
            "round_decimals": 2,
            "override_threshold": 0.18,
            "oof_mae": None,
        },
    ),
    (
        "forced_cat25",
        {
            "weights": {
                "extra_trees": 0.65,
                best_cat_name: 0.25,
                "pair_k1": 0.10,
            },
            "round_decimals": 2,
            "override_threshold": 0.18,
            "oof_mae": None,
        },
    ),
]

configs.extend(forced_configs)

# 중복 제거
unique_configs = []
seen = set()

for name, config in configs:
    key = (
        tuple(sorted(config["weights"].items())),
        config["round_decimals"],
        config["override_threshold"],
    )
    if key not in seen:
        seen.add(key)
        unique_configs.append((name, config))

configs = unique_configs[:5]

for name, config in configs:
    pred = sum(
        config["weights"][model_name] * oof[model_name]
        for model_name in config["weights"]
    )
    processed = apply_postprocess(
        pred,
        config["round_decimals"],
        config["override_threshold"],
        nearest_dist_oof,
        nearest_target_oof,
    )
    config["oof_mae"] = mean_absolute_error(y, processed)
    print(name, config)


cv_best {'weights': {'extra_trees': 0.8500000000000001, 'cat_rmse': 0.0, 'pair_k1': 0.1499999999999999}, 'round_decimals': 2, 'override_threshold': 0.2, 'oof_mae': 0.14709333333333333}
cat_10_15 {'weights': {'extra_trees': 0.8, 'cat_rmse': 0.1, 'pair_k1': 0.09999999999999995}, 'round_decimals': 2, 'override_threshold': 0.22, 'oof_mae': 0.15044333333333332}
cat_20_25 {'weights': {'extra_trees': 0.7000000000000001, 'cat_rmse': 0.2, 'pair_k1': 0.09999999999999992}, 'round_decimals': None, 'override_threshold': 0.22, 'oof_mae': 0.15394778303113407}
forced_cat15 {'weights': {'extra_trees': 0.7, 'cat_rmse': 0.15, 'pair_k1': 0.15}, 'round_decimals': 2, 'override_threshold': 0.18, 'oof_mae': 0.15229333333333334}
forced_cat25 {'weights': {'extra_trees': 0.65, 'cat_rmse': 0.25, 'pair_k1': 0.1}, 'round_decimals': 2, 'override_threshold': 0.18, 'oof_mae': 0.15603333333333333}


## 7. 전체 데이터 학습 및 제출 파일 생성


In [9]:
if not test_path.exists() or not submission_path.exists():
    print(
        "test.csv 또는 sample_submission.csv가 현재 환경에 없습니다. "
        "사용자 로컬 DATA_DIR에서 실행하면 제출 파일이 생성됩니다."
    )
else:
    test_raw = pd.read_csv(test_path)
    submission_template = pd.read_csv(submission_path)

    test_input = (
        test_raw
        .drop(columns=[ID_COLUMN])
        .reset_index(drop=True)
    )

    # ExtraTrees와 Pair
    bounds = compute_iqr_bounds(raw_input, WINSORIZE_COLUMNS)
    train_w = apply_winsorize(raw_input, bounds)
    test_w = apply_winsorize(test_input, bounds)

    train_base = add_base_features(train_w)
    test_base = add_base_features(test_w)

    print("ExtraTrees 전체 데이터 학습")
    et_matrix_test = extra_trees_prediction_matrix(
        train_base,
        test_base,
        y,
    )
    et_test_pred = np.quantile(
        et_matrix_test,
        TREE_QUANTILE,
        axis=1,
    )

    print("Pair 전체 데이터 생성")
    pair_matrix_test = pair_prediction_matrix(
        train_base,
        test_base,
        y,
    )
    pair_test_pred = np.quantile(
        pair_matrix_test,
        PAIR_QUANTILE,
        axis=1,
    )

    # CatBoost 전체 데이터 학습
    train_cat = add_catboost_features(raw_input)
    test_cat = add_catboost_features(test_input)

    # OOF best iteration 평균을 사용하되 안전한 최솟값 적용
    mean_best_iter = int(
        max(
            300,
            np.mean(best_iterations[best_cat_name]),
        )
    )

    full_config = CATBOOST_CONFIGS[best_cat_name].copy()
    full_config["iterations"] = mean_best_iter

    train_cb, test_cb, categorical_cols = prepare_native_catboost_frames(
        train_cat,
        test_cat,
    )

    print(
        f"{best_cat_name} 전체 데이터 학습, iterations={mean_best_iter}"
    )
    cat_model = CatBoostRegressor(
        **full_config,
        random_seed=RANDOM_SEED,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
    )
    cat_model.fit(
        train_cb,
        y,
        cat_features=categorical_cols,
        verbose=False,
    )
    cat_test_pred = cat_model.predict(test_cb)

    test_predictions = {
        "extra_trees": et_test_pred,
        "pair_k1": pair_test_pred,
        best_cat_name: cat_test_pred,
    }

    nearest_dist_test, nearest_target_test = nearest_neighbor_lookup(
        raw_input,
        test_input,
        y,
        numeric_cols,
    )

    saved_paths = []

    for rank, (name, config) in enumerate(configs, start=1):
        pred = sum(
            config["weights"][model_name]
            * test_predictions[model_name]
            for model_name in config["weights"]
        )

        pred = apply_postprocess(
            pred,
            config["round_decimals"],
            config["override_threshold"],
            nearest_dist_test,
            nearest_target_test,
        )

        output = submission_template.copy()
        output[TARGET] = pred

        output_path = (
            OUTPUT_DIR
            / f"submit_native_cat_v5_rank{rank}_{name}.csv"
        )
        output.to_csv(
            output_path,
            index=False,
            encoding="utf-8",
        )
        saved_paths.append(output_path)

        print("\nSaved:", output_path)
        print("Config:", config)
        print(
            "Prediction range:",
            float(pred.min()),
            float(pred.max()),
        )
        display(output.head())

    print("\n생성 완료")
    for path in saved_paths:
        print(path)


ExtraTrees 전체 데이터 학습
Pair 전체 데이터 생성
cat_rmse 전체 데이터 학습, iterations=2497

Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank1_cv_best.csv
Config: {'weights': {'extra_trees': 0.8500000000000001, 'cat_rmse': 0.0, 'pair_k1': 0.1499999999999999}, 'round_decimals': 2, 'override_threshold': 0.2, 'oof_mae': 0.14709333333333333}
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.60
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank2_cat_10_15.csv
Config: {'weights': {'extra_trees': 0.8, 'cat_rmse': 0.1, 'pair_k1': 0.09999999999999995}, 'round_decimals': 2, 'override_threshold': 0.22, 'oof_mae': 0.15044333333333332}
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.59
1,TEST_0001,0.97
2,TEST_0002,0.20
3,TEST_0003,0.50
4,TEST_0004,0.54



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank3_cat_20_25.csv
Config: {'weights': {'extra_trees': 0.7000000000000001, 'cat_rmse': 0.2, 'pair_k1': 0.09999999999999992}, 'round_decimals': None, 'override_threshold': 0.22, 'oof_mae': 0.15394778303113407}
Prediction range: 0.0 1.0


...


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank4_forced_cat15.csv
Config: {'weights': {'extra_trees': 0.7, 'cat_rmse': 0.15, 'pair_k1': 0.15}, 'round_decimals': 2, 'override_threshold': 0.18, 'oof_mae': 0.15229333333333334}
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.59
1,TEST_0001,0.97
2,TEST_0002,0.20
3,TEST_0003,0.50
4,TEST_0004,0.54



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank5_forced_cat25.csv
Config: {'weights': {'extra_trees': 0.65, 'cat_rmse': 0.25, 'pair_k1': 0.1}, 'round_decimals': 2, 'override_threshold': 0.18, 'oof_mae': 0.15603333333333333}
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.59
1,TEST_0001,0.97
2,TEST_0002,0.21
3,TEST_0003,0.49
4,TEST_0004,0.54



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank1_cv_best.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank2_cat_10_15.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank3_cat_20_25.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank4_forced_cat15.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_native_cat_v5_rank5_forced_cat25.csv
